In [ ]:
# Data and Math Libraries
import os
os.environ.setdefault('POLARS_MAX_THREADS', '1')  # avoids a torch/polars native-thread deadlock on macOS
import polars as pl
import numpy as np
from datetime import datetime, timedelta
import random

#Machine Learning Libraries
import torch
import torch.nn as nn 
import torch.optim as optim
import research 
import models

#visualization 
import altair as alt

#data sources
import binance

In [3]:
print(research.__file__)
print([x for x in dir(research) if not x.startswith('_')])

/Users/arezziorietti/Documents/ML Projects/research.py
['np', 'random', 'set_seed', 'torch']


In [4]:
research.set_seed(42)

In [7]:
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_cols(-1) 

polars.config.Config

In [ ]:
sym = 'BTCUSDT'

hist_data_window = 7 * 4 * 6

time_interval = '1h'

max_lags = 4

forecast_horizon = 1 

annualized_rate = research.sharpe_annualization_factor(time_interval, 365, 24)

# Part 1: BTCUSDT Research

## Download & Load Price Data

In [ ]:
# Download hist_data_window days of trade data (skips days already cached)
binance.download_trades(sym, hist_data_window)

In [ ]:
ts = research.load_ohlc_timeseries(sym, time_interval)
ts

In [ ]:
research.plot_static_timeseries(ts, sym, 'close', time_interval)

## Feature Engineering

Build the target (future log return) and auto-regressive lag features.

In [ ]:
target = 'close_log_return'
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=max_lags)
ts = ts.drop_nulls()
ts

In [ ]:
research.plot_distribution(ts, target, no_bins=100)

In [ ]:
# Auto-correlation between the target and its own lags
research.auto_reg_corr_matrx(ts, target, max_lags)

## Benchmark Linear Models

Try every combination of up to 3 lag features and rank by annualized Sharpe.

In [ ]:
feature_pool = [f'{target}_lag_{i}' for i in range(1, max_lags + 1)]
btc_benchmark = research.benchmark_linear_models(
    ts, target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss()
)
btc_benchmark

In [ ]:
btc_best = btc_benchmark.row(0, named=True)
btc_best_features = btc_best['features'].split(',')
btc_best_features

## Best Model: Trade-Level Performance

In [ ]:
btc_model = models.LinearModel(len(btc_best_features))
btc_model_trades = research.learn_model_trades(ts, btc_best_features, target, btc_model, loss=nn.L1Loss())
research.plot_column(btc_model_trades, 'equity_curve')

## Add Transaction Fees

In [ ]:
maker_fee = 0.00045
taker_fee = 0.00045
btc_model_trades = research.add_tx_fees_log(btc_model_trades, maker_fee, taker_fee)
research.plot_column(btc_model_trades, 'equity_curve_net_taker')

## Save Model

In [ ]:
torch.save(btc_model.state_dict(), 'BTCUSDT_model_weights.pth')
print(f"Saved BTCUSDT model with features {btc_best_features}")

# Part 2: ETHUSDT Research

Repeat the exact same research pipeline on a second asset (ETHUSDT) so we're not overfitting our conclusions to a single symbol.

## Download & Load Price Data

In [ ]:
sym = 'ETHUSDT'
binance.download_trades(sym, hist_data_window)

In [ ]:
eth_ts = research.load_ohlc_timeseries(sym, time_interval)
eth_ts

In [ ]:
research.plot_static_timeseries(eth_ts, sym, 'close', time_interval)

## Feature Engineering

In [ ]:
eth_ts = research.add_log_return_features(eth_ts, 'close', forecast_horizon, max_no_lags=max_lags)
eth_ts = eth_ts.drop_nulls()
eth_ts

In [ ]:
research.plot_distribution(eth_ts, target, no_bins=100)

In [ ]:
research.auto_reg_corr_matrx(eth_ts, target, max_lags)

## Benchmark Linear Models

In [ ]:
eth_benchmark = research.benchmark_linear_models(
    eth_ts, target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss()
)
eth_benchmark

In [ ]:
eth_best = eth_benchmark.row(0, named=True)
eth_best_features = eth_best['features'].split(',')
eth_best_features

## Best Model: Trade-Level Performance

In [ ]:
eth_model = models.LinearModel(len(eth_best_features))
eth_model_trades = research.learn_model_trades(eth_ts, eth_best_features, target, eth_model, loss=nn.L1Loss())
research.plot_column(eth_model_trades, 'equity_curve')

## Add Transaction Fees

In [ ]:
eth_model_trades = research.add_tx_fees_log(eth_model_trades, maker_fee, taker_fee)
research.plot_column(eth_model_trades, 'equity_curve_net_taker')

## Save Model

In [ ]:
torch.save(eth_model.state_dict(), 'ETHUSDT_model_weights.pth')
print(f"Saved ETHUSDT model with features {eth_best_features}")

# Part 3: BTCUSDT vs ETHUSDT Comparison

In [ ]:
comparison = pl.concat([
    btc_benchmark.head(1).with_columns(pl.lit('BTCUSDT').alias('sym')),
    eth_benchmark.head(1).with_columns(pl.lit('ETHUSDT').alias('sym')),
]).select(['sym'] + [c for c in btc_benchmark.columns if c != 'sym'])
comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(btc_model_trades['equity_curve_net_taker'], label='BTCUSDT')
plt.plot(eth_model_trades['equity_curve_net_taker'], label='ETHUSDT')
plt.title('Net (after taker fees) Equity Curve: BTCUSDT vs ETHUSDT')
plt.xlabel('Trade #')
plt.ylabel('Cumulative Log Return')
plt.legend()
plt.tight_layout()
plt.show()